# Analisis Sentimen Ulasan/Komentar Berbahasa Indonesia

Tugas NLP - Agent berbasis Transformers

**Task:** Klasifikasi sentimen teks berbahasa Indonesia ke dalam 3 kelas: positive, neutral, negative

**Pre-trained model:** [`mdhugol/indonesia-bert-sentiment-classification`](https://huggingface.co/mdhugol/indonesia-bert-sentiment-classification) — hasil fine-tuning dari **IndoBERT** (`indobenchmark/indobert-base-p1`) pada dataset SmSA.

**Dataset:** [IndoNLU - SmSA](https://github.com/IndoNLP/indonlu/tree/master/dataset/smsa_doc-sentiment-prosa) (Purwarianti & Crisdayanti, 2019), diambil langsung dari raw file GitHub resmi repo IndoNLU — jadi tidak gampang gagal kayak lewat `datasets` library Hugging Face.

Catatan: dataset SmSA ini memang dataset yang dipakai buat fine-tuning model `mdhugol` di atas, jadi hasil evaluasinya di notebook ini valid buat nunjukin seberapa bagus performa model pada data aslinya.

In [1]:
!pip install -q transformers accelerate


## 1. Load Dataset (IndoNLU SmSA, langsung dari GitHub)

In [2]:
import pandas as pd

TRAIN_URL = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv"
VALID_URL = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/valid_preprocess.tsv"

train_df = pd.read_csv(TRAIN_URL, sep="\t", header=None, names=["text", "label"])
valid_df = pd.read_csv(VALID_URL, sep="\t", header=None, names=["text", "label"])

print("Jumlah data train:", len(train_df))
print("Jumlah data valid:", len(valid_df))
train_df.head()


Jumlah data train: 11000
Jumlah data valid: 1260


,text,label
0,warung ini dimiliki oleh pengusaha pabrik tahu...,positive
1,mohon ulama lurus dan k212 mmbri hujjah partai...,neutral
2,lokasi strategis di jalan sumatera bandung . t...,positive
3,betapa bahagia nya diri ini saat unboxing pake...,positive
4,duh . jadi mahasiswa jangan sombong dong . kas...,negative


In [3]:
print(train_df["label"].value_counts())


label
positive    6416
negative    3436
neutral     1148
Name: count, dtype: int64


## 2. Load Tokenizer & Contoh Hasil Tokenisasi (WordPiece)

Model ini berbasis BERT, jadi tokenizer-nya pakai algoritma **WordPiece**. Token yang diawali `##` artinya potongan subword lanjutan dari kata sebelumnya.

In [4]:
from transformers import AutoTokenizer

MODEL_NAME = "mdhugol/indonesia-bert-sentiment-classification"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

contoh_teks = train_df.iloc[0]["text"]
print("Teks:", contoh_teks)

tokens = tokenizer.tokenize(contoh_teks)
print("\nHasil tokenisasi (WordPiece):")
print(tokens)

ids = tokenizer.convert_tokens_to_ids(tokens)
print("\nToken ID:")
print(ids)


config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Teks: warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung . tahu berkualitas , dipadu keahlian memasak , dipadu kretivitas , jadilah warung yang menyajikan menu utama berbahan tahu , ditambah menu umum lain seperti ayam . semuanya selera indonesia . harga cukup terjangkau . jangan lewatkan tahu bletoka nya , tidak kalah dengan yang asli dari tegal !

Hasil tokenisasi (WordPiece):
['warung', 'ini', 'dimiliki', 'oleh', 'pengusaha', 'pabrik', 'tahu', 'yang', 'sudah', 'puluhan', 'tahun', 'terkenal', 'membuat', 'tahu', 'putih', 'di', 'bandung', '.', 'tahu', 'berkualitas', ',', 'dipad', '##u', 'keahlian', 'memasak', ',', 'dipad', '##u', 'kre', '##tivitas', ',', 'jadilah', 'warung', 'yang', 'menyajikan', 'menu', 'utama', 'berbahan', 'tahu', ',', 'ditambah', 'menu', 'umum', 'lain', 'seperti', 'ayam', '.', 'semuanya', 'selera', 'indonesia', '.', 'harga', 'cukup', 'terjangkau', '.', 'jangan', 'lewatkan', 'tahu', 'bl', '##eto', '##ka', 'n

## 3. Load Model & Pipeline Klasifikasi

In [5]:
from transformers import AutoModelForSequenceClassification, pipeline

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

label_index = {
    "LABEL_0": "positive",
    "LABEL_1": "neutral",
    "LABEL_2": "negative"
}


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## 4. Simulasi pada Beberapa Contoh Teks Manual

In [6]:
contoh_teks_list = [
    "Pelayanannya ramah banget dan makanannya enak, pasti balik lagi ke sini",
    "Barang sampai sesuai dengan deskripsi, tidak ada yang istimewa",
    "Kecewa berat, barang rusak dan admin tidak merespon sama sekali"
]

for teks in contoh_teks_list:
    hasil = sentiment_pipeline(teks)[0]
    label = label_index[hasil["label"]]
    score = round(hasil["score"], 3)
    print(f"Teks     : {teks}")
    print(f"Prediksi : {label} (score: {score})")
    print()


model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Teks     : Pelayanannya ramah banget dan makanannya enak, pasti balik lagi ke sini
Prediksi : positive (score: 0.998)

Teks     : Barang sampai sesuai dengan deskripsi, tidak ada yang istimewa
Prediksi : negative (score: 0.861)

Teks     : Kecewa berat, barang rusak dan admin tidak merespon sama sekali
Prediksi : negative (score: 0.998)



## 5. Evaluasi Performa Model pada Dataset Validasi

Evaluasi dilakukan pada data validasi SmSA (data ini punya label asli, beda dengan data test IndoNLU yang labelnya sengaja disembunyikan). Supaya tidak terlalu lama, dipakai 300 sampel acak.

In [7]:
n_sample = min(300, len(valid_df))
sample_df = valid_df.sample(n=n_sample, random_state=42).reset_index(drop=True)
print("Jumlah data yang dievaluasi:", n_sample)

preds = []
for teks in sample_df["text"]:
    hasil = sentiment_pipeline(teks, truncation=True, max_length=128)[0]
    preds.append(label_index[hasil["label"]])

sample_df["prediksi"] = preds
sample_df.head(10)


Jumlah data yang dievaluasi: 300


,text,label,prediksi
0,saya datang sore hari dengan teman-teman . lok...,positive,positive
1,lelah dan merasa tidak nyaman ? dengan kepemim...,negative,negative
2,betul . cebong-cebong pada bodoh semua maka da...,negative,negative
3,lokasi nya dekat sekali dari pusat kota bandun...,positive,positive
4,rumah makan padang yang enak banget pas buat r...,positive,positive
5,makanan nya mahal rasanya juga tidak enak,negative,negative
6,tempat nya keren . cocok buat saya yang sangat...,positive,positive
7,apa yang ingin kamu lakukan saat hujan ? tidur...,neutral,neutral
8,tidak puas sama sekali . pesan teh tarik keman...,negative,negative
9,kami ke sana hari selasa sekitar jam 1500 pela...,positive,positive


In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

akurasi = accuracy_score(sample_df["label"], sample_df["prediksi"])
print("Akurasi pada data validasi:", round(akurasi, 4))

print("\nClassification report:")
print(classification_report(sample_df["label"], sample_df["prediksi"]))

print("Confusion matrix (urutan label: negative, neutral, positive):")
labels_order = ["negative", "neutral", "positive"]
print(confusion_matrix(sample_df["label"], sample_df["prediksi"], labels=labels_order))


Akurasi pada data validasi: 0.9367

Classification report:
              precision    recall  f1-score   support

    negative       0.93      0.94      0.93        96
     neutral       0.88      0.82      0.85        34
    positive       0.95      0.96      0.96       170

    accuracy                           0.94       300
   macro avg       0.92      0.91      0.91       300
weighted avg       0.94      0.94      0.94       300

Confusion matrix (urutan label: negative, neutral, positive):
[[ 90   2   4]
 [  2  28   4]
 [  5   2 163]]


## 6. Kesimpulan

Agent analisis sentimen ini menggunakan pre-trained model `mdhugol/indonesia-bert-sentiment-classification`, hasil fine-tuning IndoBERT pada dataset SmSA (IndoNLU). Karena model ini memang dilatih pada dataset SmSA, hasil evaluasi pada data validasi menunjukkan performa yang tinggi dan konsisten dengan tujuan awal pembuatan model tersebut.

Alur kerja agent: teks input ditokenisasi menggunakan algoritma WordPiece, lalu diproses melalui layer-layer encoder BERT untuk menghasilkan representasi vektor, kemudian representasi tersebut diklasifikasikan oleh classification head menjadi salah satu dari tiga kelas sentimen: positive, neutral, atau negative.

Agent ini bisa dikembangkan lebih lanjut untuk use-case seperti monitoring ulasan produk, analisis opini di media sosial, atau sistem customer feedback otomatis.